In [1]:
import requests, pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path
from itertools import combinations

# Сбор данных

In [2]:
BASE = 'https://swapi.dev/api/'
RESOURCES = ['people', 'films', 'planets', 'starships', 'vehicles', 'species']
RAW_DIR = Path('data/raw')

## Функция для сбора данных с сайта SWAPI

In [19]:
def fetch_all(resource):
    url = f"https://swapi.dev/api/{resource}/"
    all_results = []

    while url:
        response = requests.get(url)
        response.raise_for_status() 
        data = response.json()

        all_results.extend(data["results"])

        url = data["next"]

        print(f"{resource}: собрано {len(all_results)} из {data['count']}")
    return all_results

In [21]:
all_results = {res: fetch_all(res) for res in RESOURCES}

people: собрано 10 из 82
people: собрано 20 из 82
people: собрано 30 из 82
people: собрано 40 из 82
people: собрано 50 из 82
people: собрано 60 из 82
people: собрано 70 из 82
people: собрано 80 из 82
people: собрано 82 из 82
films: собрано 6 из 6
planets: собрано 10 из 60
planets: собрано 20 из 60
planets: собрано 30 из 60
planets: собрано 40 из 60
planets: собрано 50 из 60
planets: собрано 60 из 60
starships: собрано 10 из 36
starships: собрано 20 из 36
starships: собрано 30 из 36
starships: собрано 36 из 36
vehicles: собрано 10 из 39
vehicles: собрано 20 из 39
vehicles: собрано 30 из 39
vehicles: собрано 39 из 39
species: собрано 10 из 37
species: собрано 20 из 37
species: собрано 30 из 37
species: собрано 37 из 37


## Сохранение сырых данных

In [7]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

for resource, item in all_results.items():
    df = pd.DataFrame(item)
    df.to_json(RAW_DIR / f"{resource}.json", force_ascii=False)
    print(f"Сохранено: {resource}.json ({len(df)} записей)")

NameError: name 'all_results' is not defined

## Загрузка сырых данных

In [3]:
def load_resource(resource):
    with (RAW_DIR / f"{resource}.json").open() as f:
        return pd.read_json(f)


def load_all() -> dict[str, pd.DataFrame]:
    return {res: load_resource(res) for res in RESOURCES}

In [4]:
dfs = load_all()

In [5]:
df_people = dfs["people"]
df_films = dfs["films"]
df_planets = dfs["planets"]
df_starships = dfs["starships"]
df_vehicles = dfs["vehicles"]
df_species = dfs["species"]

In [6]:
df_films.info()

<class 'pandas.DataFrame'>
Index: 6 entries, 0 to 5
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          6 non-null      str   
 1   episode_id     6 non-null      int64 
 2   opening_crawl  6 non-null      str   
 3   director       6 non-null      str   
 4   producer       6 non-null      str   
 5   release_date   6 non-null      str   
 6   characters     6 non-null      object
 7   planets        6 non-null      object
 8   starships      6 non-null      object
 9   vehicles       6 non-null      object
 10  species        6 non-null      object
 11  created        6 non-null      str   
 12  edited         6 non-null      str   
 13  url            6 non-null      str   
dtypes: int64(1), object(5), str(8)
memory usage: 720.0+ bytes


In [7]:
df_starships.info()

<class 'pandas.DataFrame'>
Index: 36 entries, 0 to 35
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   name                    36 non-null     str   
 1   model                   36 non-null     str   
 2   manufacturer            36 non-null     str   
 3   cost_in_credits         36 non-null     str   
 4   length                  36 non-null     str   
 5   max_atmosphering_speed  36 non-null     str   
 6   crew                    36 non-null     str   
 7   passengers              36 non-null     str   
 8   cargo_capacity          36 non-null     str   
 9   consumables             36 non-null     str   
 10  hyperdrive_rating       36 non-null     str   
 11  MGLT                    36 non-null     str   
 12  starship_class          36 non-null     str   
 13  pilots                  36 non-null     object
 14  films                   36 non-null     object
 15  created                 

In [8]:
df_people.info()

<class 'pandas.DataFrame'>
Index: 82 entries, 0 to 81
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        82 non-null     str   
 1   height      82 non-null     str   
 2   mass        82 non-null     str   
 3   hair_color  82 non-null     str   
 4   skin_color  82 non-null     str   
 5   eye_color   82 non-null     str   
 6   birth_year  82 non-null     str   
 7   gender      82 non-null     str   
 8   homeworld   82 non-null     str   
 9   films       82 non-null     object
 10  species     82 non-null     object
 11  vehicles    82 non-null     object
 12  starships   82 non-null     object
 13  created     82 non-null     str   
 14  edited      82 non-null     str   
 15  url         82 non-null     str   
dtypes: object(4), str(12)
memory usage: 10.9+ KB


In [9]:
df_planets.info()

<class 'pandas.DataFrame'>
Index: 60 entries, 0 to 59
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   name             60 non-null     str   
 1   rotation_period  60 non-null     str   
 2   orbital_period   60 non-null     str   
 3   diameter         60 non-null     str   
 4   climate          60 non-null     str   
 5   gravity          60 non-null     str   
 6   terrain          60 non-null     str   
 7   surface_water    60 non-null     str   
 8   population       60 non-null     str   
 9   residents        60 non-null     object
 10  films            60 non-null     object
 11  created          60 non-null     str   
 12  edited           60 non-null     str   
 13  url              60 non-null     str   
dtypes: object(2), str(12)
memory usage: 7.0+ KB


In [10]:
df_species.info()

<class 'pandas.DataFrame'>
Index: 37 entries, 0 to 36
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   name              37 non-null     str   
 1   classification    37 non-null     str   
 2   designation       37 non-null     str   
 3   average_height    37 non-null     str   
 4   skin_colors       37 non-null     str   
 5   hair_colors       37 non-null     str   
 6   eye_colors        37 non-null     str   
 7   average_lifespan  37 non-null     str   
 8   homeworld         36 non-null     str   
 9   language          37 non-null     str   
 10  people            37 non-null     object
 11  films             37 non-null     object
 12  created           37 non-null     str   
 13  edited            37 non-null     str   
 14  url               37 non-null     str   
dtypes: object(2), str(13)
memory usage: 4.6+ KB


In [11]:
df_vehicles.info()

<class 'pandas.DataFrame'>
Index: 39 entries, 0 to 38
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   name                    39 non-null     str   
 1   model                   39 non-null     str   
 2   manufacturer            39 non-null     str   
 3   cost_in_credits         39 non-null     str   
 4   length                  39 non-null     str   
 5   max_atmosphering_speed  39 non-null     str   
 6   crew                    39 non-null     int64 
 7   passengers              39 non-null     str   
 8   cargo_capacity          39 non-null     str   
 9   consumables             39 non-null     str   
 10  vehicle_class           39 non-null     str   
 11  pilots                  39 non-null     object
 12  films                   39 non-null     object
 13  created                 39 non-null     str   
 14  edited                  39 non-null     str   
 15  url                     

In [12]:
df_films

,title,episode_id,opening_crawl,director,producer,release_date,characters,planets,starships,vehicles,species,created,edited,url
0,A New Hope,4,It is a period of civil war.\r\nRebel spaceshi...,George Lucas,"Gary Kurtz, Rick McCallum",1977-05-25,"[https://swapi.dev/api/people/1/, https://swap...","[https://swapi.dev/api/planets/1/, https://swa...","[https://swapi.dev/api/starships/2/, https://s...","[https://swapi.dev/api/vehicles/4/, https://sw...","[https://swapi.dev/api/species/1/, https://swa...",2014-12-10T14:23:31.880000Z,2014-12-20T19:49:45.256000Z,https://swapi.dev/api/films/1/
1,The Empire Strikes Back,5,It is a dark time for the\r\nRebellion. Althou...,Irvin Kershner,"Gary Kurtz, Rick McCallum",1980-05-17,"[https://swapi.dev/api/people/1/, https://swap...","[https://swapi.dev/api/planets/4/, https://swa...","[https://swapi.dev/api/starships/3/, https://s...","[https://swapi.dev/api/vehicles/8/, https://sw...","[https://swapi.dev/api/species/1/, https://swa...",2014-12-12T11:26:24.656000Z,2014-12-15T13:07:53.386000Z,https://swapi.dev/api/films/2/
2,Return of the Jedi,6,Luke Skywalker has returned to\r\nhis home pla...,Richard Marquand,"Howard G. Kazanjian, George Lucas, Rick McCallum",1983-05-25,"[https://swapi.dev/api/people/1/, https://swap...","[https://swapi.dev/api/planets/1/, https://swa...","[https://swapi.dev/api/starships/2/, https://s...","[https://swapi.dev/api/vehicles/8/, https://sw...","[https://swapi.dev/api/species/1/, https://swa...",2014-12-18T10:39:33.255000Z,2014-12-20T09:48:37.462000Z,https://swapi.dev/api/films/3/
3,The Phantom Menace,1,Turmoil has engulfed the\r\nGalactic Republic....,George Lucas,Rick McCallum,1999-05-19,"[https://swapi.dev/api/people/2/, https://swap...","[https://swapi.dev/api/planets/1/, https://swa...","[https://swapi.dev/api/starships/31/, https://...","[https://swapi.dev/api/vehicles/33/, https://s...","[https://swapi.dev/api/species/1/, https://swa...",2014-12-19T16:52:55.740000Z,2014-12-20T10:54:07.216000Z,https://swapi.dev/api/films/4/
4,Attack of the Clones,2,There is unrest in the Galactic\r\nSenate. Sev...,George Lucas,Rick McCallum,2002-05-16,"[https://swapi.dev/api/people/2/, https://swap...","[https://swapi.dev/api/planets/1/, https://swa...","[https://swapi.dev/api/starships/21/, https://...","[https://swapi.dev/api/vehicles/4/, https://sw...","[https://swapi.dev/api/species/1/, https://swa...",2014-12-20T10:57:57.886000Z,2014-12-20T20:18:48.516000Z,https://swapi.dev/api/films/5/
5,Revenge of the Sith,3,War! The Republic is crumbling\r\nunder attack...,George Lucas,Rick McCallum,2005-05-19,"[https://swapi.dev/api/people/1/, https://swap...","[https://swapi.dev/api/planets/1/, https://swa...","[https://swapi.dev/api/starships/2/, https://s...","[https://swapi.dev/api/vehicles/33/, https://s...","[https://swapi.dev/api/species/1/, https://swa...",2014-12-20T18:49:38.403000Z,2014-12-20T20:47:52.073000Z,https://swapi.dev/api/films/6/


# Предобработка данных

### В данных таблицах, есть особенность с некоторыми колонками. Например, датафрейме films есть колонка characters. Значения этих колонок имею список, а не значение. Следовательно нужно преобразовать в id-номера, чтобы можно было корректно делать соединение таблиц

In [13]:
def extract_id(value):
    if not isinstance(value, str):
        return None
    
    match = re.search(r"/(\d+)/?$", value)
    return int(match.group(1)) if match else None


def to_ids(value):
    if isinstance(value, list):
        return [extract_id(u) for u in value]
    return extract_id(value)

In [14]:
def check_type_columns(df):
    list_cols, single_cols = [], []
    for col in df.columns:
        if col == 'url':
            continue
        
        rows = df[col].tolist()
        for row in rows:
            if isinstance(row, list) and row and "swapi.dev" in row[0]:
                list_cols.append(col)
                break
            if isinstance(row, str) and "swapi.dev" in row:
                single_cols.append(col)
                break
    return list_cols + single_cols

In [15]:
def normalize_columns(df: pd.DataFrame, id_col):
    df = df.copy()
    df[id_col] = df["url"].apply(extract_id) 
    list_edit_cols = check_type_columns(df)
    for column in list_edit_cols:
        df[column] = df[column].apply(to_ids)
    return df

In [16]:
DICT_RESOURCE_TO_ID = {
    "people": "people_id",
    "films": "film_id",
    "planets": "planet_id",
    "starships": "starship_id",
    "vehicles": "vehicle_id",
    "species": "species_id",
}

df_people_norm = normalize_columns(df_people, DICT_RESOURCE_TO_ID['people'])
df_films_norm = normalize_columns(df_films, DICT_RESOURCE_TO_ID['films'])
df_planets_norm = normalize_columns(df_planets, DICT_RESOURCE_TO_ID['planets'])
df_species_norm = normalize_columns(df_species, DICT_RESOURCE_TO_ID['species'])
df_starships_norm = normalize_columns(df_starships, DICT_RESOURCE_TO_ID['starships'])
df_vehicles_norm = normalize_columns(df_vehicles, DICT_RESOURCE_TO_ID['vehicles'])

In [17]:
df_people_norm.head()

,name,height,mass,hair_color,skin_color,eye_color,birth_year,gender,homeworld,films,species,vehicles,starships,created,edited,url,people_id
0,Luke Skywalker,172,77,blond,fair,blue,19BBY,male,1,"[1, 2, 3, 6]",[],"[14, 30]","[12, 22]",2014-12-09T13:50:51.644000Z,2014-12-20T21:17:56.891000Z,https://swapi.dev/api/people/1/,1
1,C-3PO,167,75,n/a,gold,yellow,112BBY,n/a,1,"[1, 2, 3, 4, 5, 6]",[2],[],[],2014-12-10T15:10:51.357000Z,2014-12-20T21:17:50.309000Z,https://swapi.dev/api/people/2/,2
2,R2-D2,96,32,n/a,"white, blue",red,33BBY,n/a,8,"[1, 2, 3, 4, 5, 6]",[2],[],[],2014-12-10T15:11:50.376000Z,2014-12-20T21:17:50.311000Z,https://swapi.dev/api/people/3/,3
3,Darth Vader,202,136,none,white,yellow,41.9BBY,male,1,"[1, 2, 3, 6]",[],[],[13],2014-12-10T15:18:20.704000Z,2014-12-20T21:17:50.313000Z,https://swapi.dev/api/people/4/,4
4,Leia Organa,150,49,brown,light,brown,19BBY,female,2,"[1, 2, 3, 6]",[],[30],[],2014-12-10T15:20:09.791000Z,2014-12-20T21:17:50.315000Z,https://swapi.dev/api/people/5/,5


In [18]:
df_films_norm.head()

,title,episode_id,opening_crawl,director,producer,release_date,characters,planets,starships,vehicles,species,created,edited,url,film_id
0,A New Hope,4,It is a period of civil war.\r\nRebel spaceshi...,George Lucas,"Gary Kurtz, Rick McCallum",1977-05-25,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15...","[1, 2, 3]","[2, 3, 5, 9, 10, 11, 12, 13]","[4, 6, 7, 8]","[1, 2, 3, 4, 5]",2014-12-10T14:23:31.880000Z,2014-12-20T19:49:45.256000Z,https://swapi.dev/api/films/1/,1
1,The Empire Strikes Back,5,It is a dark time for the\r\nRebellion. Althou...,Irvin Kershner,"Gary Kurtz, Rick McCallum",1980-05-17,"[1, 2, 3, 4, 5, 10, 13, 14, 18, 20, 21, 22, 23...","[4, 5, 6, 27]","[3, 10, 11, 12, 15, 17, 21, 22, 23]","[8, 14, 16, 18, 19, 20]","[1, 2, 3, 6, 7]",2014-12-12T11:26:24.656000Z,2014-12-15T13:07:53.386000Z,https://swapi.dev/api/films/2/,2
2,Return of the Jedi,6,Luke Skywalker has returned to\r\nhis home pla...,Richard Marquand,"Howard G. Kazanjian, George Lucas, Rick McCallum",1983-05-25,"[1, 2, 3, 4, 5, 10, 13, 14, 16, 18, 20, 21, 22...","[1, 5, 7, 8, 9]","[2, 3, 10, 11, 12, 15, 17, 22, 23, 27, 28, 29]","[8, 16, 18, 19, 24, 25, 26, 30]","[1, 2, 3, 5, 6, 8, 9, 10, 15]",2014-12-18T10:39:33.255000Z,2014-12-20T09:48:37.462000Z,https://swapi.dev/api/films/3/,3
3,The Phantom Menace,1,Turmoil has engulfed the\r\nGalactic Republic....,George Lucas,Rick McCallum,1999-05-19,"[2, 3, 10, 11, 16, 20, 21, 32, 33, 34, 35, 36,...","[1, 8, 9]","[31, 32, 39, 40, 41]","[33, 34, 35, 36, 37, 38, 42]","[1, 2, 6, 11, 12, 13, 14, 15, 16, 17, 18, 19, ...",2014-12-19T16:52:55.740000Z,2014-12-20T10:54:07.216000Z,https://swapi.dev/api/films/4/,4
4,Attack of the Clones,2,There is unrest in the Galactic\r\nSenate. Sev...,George Lucas,Rick McCallum,2002-05-16,"[2, 3, 6, 7, 10, 11, 20, 21, 22, 33, 35, 36, 4...","[1, 8, 9, 10, 11]","[21, 32, 39, 43, 47, 48, 49, 52, 58]","[4, 44, 45, 46, 50, 51, 53, 54, 55, 56, 57]","[1, 2, 6, 12, 13, 15, 28, 29, 30, 31, 32, 33, ...",2014-12-20T10:57:57.886000Z,2014-12-20T20:18:48.516000Z,https://swapi.dev/api/films/5/,5


In [19]:
df_planets_norm.head()

,name,rotation_period,orbital_period,diameter,climate,gravity,terrain,surface_water,population,residents,films,created,edited,url,planet_id
0,Tatooine,23,304,10465,arid,1 standard,desert,1,200000,"[1, 2, 4, 6, 7, 8, 9, 11, 43, 62]","[1, 3, 4, 5, 6]",2014-12-09T13:50:49.641000Z,2014-12-20T20:58:18.411000Z,https://swapi.dev/api/planets/1/,1
1,Alderaan,24,364,12500,temperate,1 standard,"grasslands, mountains",40,2000000000,"[5, 68, 81]","[1, 6]",2014-12-10T11:35:48.479000Z,2014-12-20T20:58:18.420000Z,https://swapi.dev/api/planets/2/,2
2,Yavin IV,24,4818,10200,"temperate, tropical",1 standard,"jungle, rainforests",8,1000,[],[1],2014-12-10T11:37:19.144000Z,2014-12-20T20:58:18.421000Z,https://swapi.dev/api/planets/3/,3
3,Hoth,23,549,7200,frozen,1.1 standard,"tundra, ice caves, mountain ranges",100,unknown,[],[2],2014-12-10T11:39:13.934000Z,2014-12-20T20:58:18.423000Z,https://swapi.dev/api/planets/4/,4
4,Dagobah,23,341,8900,murky,N/A,"swamp, jungles",8,unknown,[],"[2, 3, 6]",2014-12-10T11:42:22.590000Z,2014-12-20T20:58:18.425000Z,https://swapi.dev/api/planets/5/,5


In [20]:
df_species_norm.head()

,name,classification,designation,average_height,skin_colors,hair_colors,eye_colors,average_lifespan,homeworld,language,people,films,created,edited,url,species_id
0,Human,mammal,sentient,180,"caucasian, black, asian, hispanic","blonde, brown, black, red","brown, blue, green, hazel, grey, amber",120,9.0,Galactic Basic,"[66, 67, 68, 74]","[1, 2, 3, 4, 5, 6]",2014-12-10T13:52:11.567000Z,2014-12-20T21:36:42.136000Z,https://swapi.dev/api/species/1/,1
1,Droid,artificial,sentient,n/a,n/a,n/a,n/a,indefinite,NaN,n/a,"[2, 3, 8, 23]","[1, 2, 3, 4, 5, 6]",2014-12-10T15:16:16.259000Z,2014-12-20T21:36:42.139000Z,https://swapi.dev/api/species/2/,2
2,Wookie,mammal,sentient,210,gray,"black, brown","blue, green, yellow, brown, golden, red",400,14.0,Shyriiwook,"[13, 80]","[1, 2, 3, 6]",2014-12-10T16:44:31.486000Z,2014-12-20T21:36:42.142000Z,https://swapi.dev/api/species/3/,3
3,Rodian,sentient,reptilian,170,"green, blue",n/a,black,unknown,23.0,Galatic Basic,[15],[1],2014-12-10T17:05:26.471000Z,2014-12-20T21:36:42.144000Z,https://swapi.dev/api/species/4/,4
4,Hutt,gastropod,sentient,300,"green, brown, tan",n/a,"yellow, red",1000,24.0,Huttese,[16],"[1, 3]",2014-12-10T17:12:50.410000Z,2014-12-20T21:36:42.146000Z,https://swapi.dev/api/species/5/,5


In [21]:
df_starships_norm.head()

,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,hyperdrive_rating,MGLT,starship_class,pilots,films,created,edited,url,starship_id
0,CR90 corvette,CR90 corvette,Corellian Engineering Corporation,3500000,150,950,30-165,600,3000000,1 year,2.0,60,corvette,[],"[1, 3, 6]",2014-12-10T14:20:33.369000Z,2014-12-20T21:23:49.867000Z,https://swapi.dev/api/starships/2/,2
1,Star Destroyer,Imperial I-class Star Destroyer,Kuat Drive Yards,150000000,"1,600",975,"47,060",n/a,36000000,2 years,2.0,60,Star Destroyer,[],"[1, 2, 3]",2014-12-10T15:08:19.848000Z,2014-12-20T21:23:49.870000Z,https://swapi.dev/api/starships/3/,3
2,Sentinel-class landing craft,Sentinel-class landing craft,"Sienar Fleet Systems, Cyngus Spaceworks",240000,38,1000,5,75,180000,1 month,1.0,70,landing craft,[],[1],2014-12-10T15:48:00.586000Z,2014-12-20T21:23:49.873000Z,https://swapi.dev/api/starships/5/,5
3,Death Star,DS-1 Orbital Battle Station,"Imperial Department of Military Research, Sien...",1000000000000,120000,n/a,"342,953","843,342",1000000000000,3 years,4.0,10,Deep Space Mobile Battlestation,[],[1],2014-12-10T16:36:50.509000Z,2014-12-20T21:26:24.783000Z,https://swapi.dev/api/starships/9/,9
4,Millennium Falcon,YT-1300 light freighter,Corellian Engineering Corporation,100000,34.37,1050,4,6,100000,2 months,0.5,75,Light freighter,"[13, 14, 25, 31]","[1, 2, 3]",2014-12-10T16:59:45.094000Z,2014-12-20T21:23:49.880000Z,https://swapi.dev/api/starships/10/,10


In [22]:
df_vehicles_norm.head()

,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,vehicle_class,pilots,films,created,edited,url,vehicle_id
0,Sand Crawler,Digger Crawler,Corellia Mining Corporation,150000,36.8,30,46,30,50000,2 months,wheeled,[],"[1, 5]",2014-12-10T15:36:25.724000Z,2014-12-20T21:30:21.661000Z,https://swapi.dev/api/vehicles/4/,4
1,T-16 skyhopper,T-16 skyhopper,Incom Corporation,14500,10.4,1200,1,1,50,0,repulsorcraft,[],[1],2014-12-10T16:01:52.434000Z,2014-12-20T21:30:21.665000Z,https://swapi.dev/api/vehicles/6/,6
2,X-34 landspeeder,X-34 landspeeder,SoroSuub Corporation,10550,3.4,250,1,1,5,unknown,repulsorcraft,[],[1],2014-12-10T16:13:52.586000Z,2014-12-20T21:30:21.668000Z,https://swapi.dev/api/vehicles/7/,7
3,TIE/LN starfighter,Twin Ion Engine/Ln Starfighter,Sienar Fleet Systems,unknown,6.4,1200,1,0,65,2 days,starfighter,[],"[1, 2, 3]",2014-12-10T16:33:52.860000Z,2014-12-20T21:30:21.670000Z,https://swapi.dev/api/vehicles/8/,8
4,Snowspeeder,t-47 airspeeder,Incom corporation,unknown,4.5,650,2,0,10,none,airspeeder,"[1, 18]",[2],2014-12-15T12:22:12Z,2014-12-20T21:30:21.672000Z,https://swapi.dev/api/vehicles/14/,14


### Создание столбцов и связок между таблицами

In [61]:
DICT_COLUMN_RESOURCES = {
    'homeworld': 'planets', 
    'planets':'planets',
    'films':'films',
    'species':'species',
    'vehicles':'vehicles',
    'starships':'starships',
    'characters':'people',
    'residents':'people',
    'people':'people',
    'pilots':'people'
}

In [62]:
def build_relations(df: pd.DataFrame, parent_resource, link_cols):
    parent_id = DICT_RESOURCE_TO_ID[parent_resource]
    relations = {}

    for col in link_cols:
        child_resource = DICT_COLUMN_RESOURCES.get(col)
        print(child_resource)
        if not child_resource:
            continue

        child_id = DICT_RESOURCE_TO_ID[child_resource]

        table = (
            df[[parent_id, col]]
            .explode(col)
            .rename(columns={col: child_id})
            .dropna(subset=[child_id])
            .reset_index(drop=True)
        )

        table_name = f"{parent_resource}_{child_resource}"
        relations[table_name] = table

    return relations

In [75]:
all_rels = []

rels_people = build_relations(
    df_people_norm, "people",
    link_cols=["films", "species", "starships", "vehicles"]
)
all_rels.append(rels_people)

rels_films = build_relations(
    df_films_norm, "films",
    link_cols=["characters", "planets", "starships", "vehicles", "species"]
)
all_rels.append(rels_films)

rels_planets = build_relations(
    df_planets_norm, "planets",
    link_cols=["residents", "films"]
)
all_rels.append(rels_planets)

rels_species = build_relations(
    df_species_norm, "species",
    link_cols=["people", "films"]
)
all_rels.append(rels_species)

rels_starships = build_relations(
    df_starships_norm, "starships",
    link_cols=["pilots", "films"]
)
all_rels.append(rels_starships)

rels_vehicles = build_relations(
    df_vehicles_norm, "vehicles",
    link_cols=["pilots", "films"]
)
all_rels.append(rels_vehicles)

films
species
starships
vehicles
people
planets
starships
vehicles
species
people
films
people
films
people
films
people
films


### Смотрю дублирующие элементы и удаляю их

In [76]:
all_name_columns = []
for rels in all_rels:
    for key in rels.keys():
        all_name_columns.append(key)

In [77]:
"_".join(all_name_columns[0].split("_")[::-1])

'films_people'

In [78]:
lst_for_remove_columns = []
for key_1, key_2 in combinations(all_name_columns, 2):
    if key_1 == "_".join(key_2.split("_")[::-1]):
        lst_for_remove_columns.append(key_2)

In [79]:
lst_for_remove_columns

['films_people',
 'species_people',
 'starships_people',
 'vehicles_people',
 'planets_films',
 'starships_films',
 'vehicles_films',
 'species_films']

In [ ]:
### Уда

In [81]:
remove = set(lst_for_remove_columns)
for i, rels in enumerate(all_rels):
    all_rels[i] = {k: v for k, v in rels.items() if k not in remove}

In [90]:
all_rels[:] = [rels for rels in all_rels if rels]